## Loading the dataset

In [1]:
import pandas as pd
data=pd.read_csv("/content/tweets.csv")
data

,id,label,tweet
0,1,0,#fingerprint #Pregnancy Test https://goo.gl/h1...
1,2,0,Finally a transparant silicon case ^^ Thanks t...
2,3,0,We love this! Would you go? #talk #makememorie...
3,4,0,I'm wired I know I'm George I was made that wa...
4,5,1,What amazing service! Apple won't even talk to...
...,...,...,...
7915,7916,0,Live out loud #lol #liveoutloud #selfie #smile...
7916,7917,0,We would like to wish you an amazing day! Make...
7917,7918,0,Helping my lovely 90 year old neighbor with he...
7918,7919,0,Finally got my #smart #pocket #wifi stay conne...


In [2]:
# Data information

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7920 entries, 0 to 7919
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      7920 non-null   int64 
 1   label   7920 non-null   int64 
 2   tweet   7920 non-null   object
dtypes: int64(2), object(1)
memory usage: 185.8+ KB


In [3]:
# Removing the unwanted feature 'id'

df=data.drop('id',axis=1)

In [4]:
df

,label,tweet
0,0,#fingerprint #Pregnancy Test https://goo.gl/h1...
1,0,Finally a transparant silicon case ^^ Thanks t...
2,0,We love this! Would you go? #talk #makememorie...
3,0,I'm wired I know I'm George I was made that wa...
4,1,What amazing service! Apple won't even talk to...
...,...,...
7915,0,Live out loud #lol #liveoutloud #selfie #smile...
7916,0,We would like to wish you an amazing day! Make...
7917,0,Helping my lovely 90 year old neighbor with he...
7918,0,Finally got my #smart #pocket #wifi stay conne...


## Data Preprocessing

In [6]:
# Removing punctuations, stop words, unnecessary patterns such as https or hashtag

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words=set(stopwords.words('english'))

def clean_text(text):
    text=text.lower()
    text=re.sub(r'http\S+|@\w+|#\w+', '', text)
    text=re.sub(r'[^a-z\s]', '', text)
    words=text.split()
    words=[w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text']=df['tweet'].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
# Cleaner version of the tweets

df

,label,tweet,clean_text
0,0,#fingerprint #Pregnancy Test https://goo.gl/h1...,test
1,0,Finally a transparant silicon case ^^ Thanks t...,finally transparant silicon case thanks uncle
2,0,We love this! Would you go? #talk #makememorie...,love would go
3,0,I'm wired I know I'm George I was made that wa...,im wired know im george made way
4,1,What amazing service! Apple won't even talk to...,amazing service apple wont even talk question ...
...,...,...,...
7915,0,Live out loud #lol #liveoutloud #selfie #smile...,live loud
7916,0,We would like to wish you an amazing day! Make...,would like wish amazing day make every minute ...
7917,0,Helping my lovely 90 year old neighbor with he...,helping lovely year old neighbor ipad morning ...
7918,0,Finally got my #smart #pocket #wifi stay conne...,finally got stay connected anytimeanywhere


In [8]:
# Label 0 is positive and label 1 is negative, checking the count for both the values

df['label'].value_counts() # 5894 positive sentiment and 2026 negative sentiment

,count
label,
0,5894
1,2026


In [9]:
# Dropping any rows with missing tweets, labels and clean_text

df=df.dropna(subset=['tweet','label','clean_text'])

In [10]:
df

,label,tweet,clean_text
0,0,#fingerprint #Pregnancy Test https://goo.gl/h1...,test
1,0,Finally a transparant silicon case ^^ Thanks t...,finally transparant silicon case thanks uncle
2,0,We love this! Would you go? #talk #makememorie...,love would go
3,0,I'm wired I know I'm George I was made that wa...,im wired know im george made way
4,1,What amazing service! Apple won't even talk to...,amazing service apple wont even talk question ...
...,...,...,...
7915,0,Live out loud #lol #liveoutloud #selfie #smile...,live loud
7916,0,We would like to wish you an amazing day! Make...,would like wish amazing day make every minute ...
7917,0,Helping my lovely 90 year old neighbor with he...,helping lovely year old neighbor ipad morning ...
7918,0,Finally got my #smart #pocket #wifi stay conne...,finally got stay connected anytimeanywhere


## Model Training

In [12]:
# Preparing the features and the target

X=df['clean_text']
y=df['label']

In [14]:
# Splitting the data

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# We limit to the top 5000 features and remove standard English stop words

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec=vectorizer.fit_transform(X_train)
X_test_vec=vectorizer.transform(X_test)

In [17]:
# Using the logisitic regression model

from sklearn.linear_model import LogisticRegression
model=LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [20]:
# Model Evaluation

from sklearn.metrics import accuracy_score, classification_report
y_pred=model.predict(X_test_vec)
accuracy=accuracy_score(y_test, y_pred)

## Accuracy and Classification report

In [21]:
print(f"Accuracy: {accuracy:.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8365

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.95      0.89      1152
           1       0.80      0.53      0.64       432

    accuracy                           0.84      1584
   macro avg       0.82      0.74      0.77      1584
weighted avg       0.83      0.84      0.83      1584



## Predicting an unseen tweet

In [23]:
# Example 1

new_tweet=["Just upgraded to the new Samsung Galaxy and the camera quality is absolutely mind-blowing! Finally taking decent pictures at night. 📸✨ #Samsung #Galaxy #photography #tech"]

new_vec=vectorizer.transform(new_tweet)
prediction=model.predict(new_vec)
print("Predicted Sentiment:", prediction[0])

# The results showed 0 which is a positive sentiment

Predicted Sentiment: 0


In [30]:
# Example 2

new_tweet=["My iPhone battery is completely draining in less than two hours after the latest iOS update. So frustrating when you actually need your phone for work! 😡 #Apple #iPhoneIssues #iOSupdate"]

new_vec=vectorizer.transform(new_tweet)
prediction=model.predict(new_vec)
print("Predicted Sentiment:", prediction[0])

# The results showed 1 which is a negative sentiment

Predicted Sentiment: 1
